In [1]:
%pip install --upgrade --quiet  xformers --quiet
%pip install --upgrade --quiet  langchain    --quiet
%pip install --upgrade --quiet  bitsandbytes --quiet
%pip install --upgrade --quiet  python-dotenv --quiet
%pip install accelerate --quiet
%pip install weaviate-client --quiet
%pip install sentence-transformers --quiet
%pip install -qU langchain-anthropic langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.2/218.2 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━

In [18]:
import weaviate
from dotenv import load_dotenv, find_dotenv
import os
from langchain.vectorstores.weaviate import Weaviate
from langchain.embeddings.huggingface import HuggingFaceEmbeddings


load_dotenv(find_dotenv("tokens.env"))
WCS_API_KEY = os.getenv("YOUR_WEAVIATE_KEY")
WCS_CLUSTER_URL = os.getenv("YOUR_WEAVIATE_CLUSTER")

client = weaviate.Client(
    url=WCS_CLUSTER_URL,
    auth_client_secret=weaviate.auth.AuthApiKey(WCS_API_KEY),
)
device = "cpu"
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"device": device, "batch_size": 32},
)


def get_index_name(client):
    return client.data_object.get()["objects"][0]["class"]


vectorstore = Weaviate(
    client,
    index_name=get_index_name(client),
    embedding=embed_model,
    text_key="text",
    by_text=False,
    attributes=["title", "authors", "pmid_id", "journal", "date"],
)
query = "children with benign childhood epilepsy"
docs = vectorstore.similarity_search_with_score(query, k=1)
print(docs[0][0].metadata)

/Users/henrismidt/anaconda3/envs/ms/lib/python3.8/site-packages/weaviate/warnings.py:158: DeprecationWarning: Dep016: You are using the Weaviate v3 client, which is deprecated.
            Consider upgrading to the new and improved v4 client instead!
            See here for usage: https://weaviate.io/developers/weaviate/client-libraries/python
            
  warnings.warn(


{'_additional': {'vector': [-0.003304616, 0.0036505037, 0.005043921, 0.072290935, -0.04269791, 0.06180549, 0.0560885, 0.030591587, 0.015711447, -0.02670391, 0.024674203, -0.024723373, 0.05747739, 0.017496387, -0.016508657, -0.037786607, -0.024544483, 0.0024251954, -0.0044264044, -0.0044447077, 0.00047288745, -0.007375094, 0.042587467, -0.007882351, 0.023320604, 0.0068156715, 0.020889131, 0.008847721, -0.019602442, -0.14737989, 0.04010915, 0.0033027977, -0.068516485, -0.06851564, -0.04417348, 0.003002709, 0.011794381, 0.05867011, -0.036564484, 0.009849701, 0.032116394, 0.012033861, 0.008409643, -0.0646204, -0.039139, -0.010597049, -0.045227922, -0.03069408, 0.08280152, -0.010391197, -0.014354953, -0.023944438, 0.034418732, 0.12874119, -0.056794353, 0.035688203, 0.046921805, -0.031094298, 0.0063471887, 0.037844762, 0.008217789, 0.058861323, -0.13968098, 0.020801302, 0.042202488, -0.027382957, -0.036475595, -0.03795098, 5.940836e-05, 0.022397729, -0.03209127, 0.033752788, -0.019979188, 0.

In [24]:
doc = docs[0]
whatever = doc[0]
print(whatever.metadata)
print(doc[1])
print(whatever.page_content)
print(whatever.metadata["authors"])

{'_additional': {'vector': [-0.003304616, 0.0036505037, 0.005043921, 0.072290935, -0.04269791, 0.06180549, 0.0560885, 0.030591587, 0.015711447, -0.02670391, 0.024674203, -0.024723373, 0.05747739, 0.017496387, -0.016508657, -0.037786607, -0.024544483, 0.0024251954, -0.0044264044, -0.0044447077, 0.00047288745, -0.007375094, 0.042587467, -0.007882351, 0.023320604, 0.0068156715, 0.020889131, 0.008847721, -0.019602442, -0.14737989, 0.04010915, 0.0033027977, -0.068516485, -0.06851564, -0.04417348, 0.003002709, 0.011794381, 0.05867011, -0.036564484, 0.009849701, 0.032116394, 0.012033861, 0.008409643, -0.0646204, -0.039139, -0.010597049, -0.045227922, -0.03069408, 0.08280152, -0.010391197, -0.014354953, -0.023944438, 0.034418732, 0.12874119, -0.056794353, 0.035688203, 0.046921805, -0.031094298, 0.0063471887, 0.037844762, 0.008217789, 0.058861323, -0.13968098, 0.020801302, 0.042202488, -0.027382957, -0.036475595, -0.03795098, 5.940836e-05, 0.022397729, -0.03209127, 0.033752788, -0.019979188, 0.

In [5]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os
from dotenv import load_dotenv
import transformers
from torch import cuda, bfloat16

# load environment variables from .env file
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv("./tokens.env"))

bitsAndBites_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16,
)

model_id = "meta-llama/Llama-2-13b-chat-hf"
hf_auth = os.environ.get("HF_AUTH")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=hf_auth)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bitsAndBites_config,
    device_map="auto",
    do_sample=True,
    token=hf_auth,
)

model.eval()

pipe = pipeline(
    task="text-generation",
    model=model,
    return_full_text=True,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01,
    repetition_penalty=1.1,
)

hf = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.10/dist-packages/transformers/models/auto/tokenization_auto.py:671: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/9.90G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/6.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [17]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
# wiki = WikipediaRetriever(top_k_results=6, doc_content_chars_max=2000)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You're a helpful AI assistant. Given a user question and some pubmed article snippets, answer the user question using only the information contained in the article snippets. Important: Name the title of the article snippet you used to generate the answer. If none of the articles answer the question, just say you don't know.\n\nHere are the pubmed articles:{context}",
        ),
        ("human", "{question}"),
    ]
)
prompt.pretty_print()

================================ System Message ================================

You're a helpful AI assistant. Given a user question and some pubmed article snippets, answer the user question using only the information contained in the article snippets. Important: Name the title of the article snippet you used to generate the answer. If none of the articles answer the question, just say you don't know.

Here are the pubmed articles:{context}

================================ Human Message =================================

{question}


In [18]:
from operator import itemgetter
from typing import List

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)


def format_docs(docs: List[Document]) -> str:
    """Convert Documents to a single string.:"""
    formatted = [
        f"Article Title: {doc.metadata['title']}\nArticle Snippet: {doc.page_content}"
        for doc in docs
    ]
    return "\n\n" + "\n\n".join(formatted)


format = itemgetter("docs") | RunnableLambda(format_docs)
# subchain for generating an answer once we've done retrieval
answer = prompt | hf | StrOutputParser()
# complete chain that calls wiki -> formats docs to string -> runs answer subchain -> returns just the answer and retrieved docs.
chain = (
    RunnableParallel(question=RunnablePassthrough(), docs=vectorstore.as_retriever())
    .assign(context=format)
    .assign(answer=answer)
    .pick(["answer", "docs"])
)

In [19]:
chain.invoke(
    "what are the most common forms of benign epilepsy syndromes for children?"
)

{'answer': '\nAssistant: Based on the article "Common and Distinctive Patterns of Cognitive Dysfunction in Children With Benign Epilepsy Syndromes," childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes are the most common forms of benign epilepsy syndromes for children.',
 'docs': [Document(page_content='Childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes are the most common forms of benign epilepsy syndromes. Although cognitive dysfunctions occur in children with both childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes, the similarity between their patterns of underlying cognitive impairments is not well understood. To describe these patterns, we examined multiple cognitive functions in children with childhood absence epilepsy and benign childhood epilepsy with centrotemporal spikes.', metadata={'authors': ['Cheng, Dazhi', 'Yan, Xiuxian', 'Gao, Zhijie', 'Xu, Keming', 'Zhou, Xinlin', 'Chen, Q